In [5]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]   # repo root
sys.path.insert(0, str(ROOT))

In [6]:
import sys
from pathlib import Path

from src.data.build_clean_df import build_divvy_datasets
from src.features.feature_engineering import add_time_and_lag_features

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

## Building the dataset

In [7]:
ROOT = Path.cwd().parents[0]

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# 1) Build demand dataset (hourly + weather)
demand = build_divvy_datasets(
    trip_paths=[
        DATA_RAW / "Divvy_Trips_2017_Q1.csv",
        DATA_RAW / "Divvy_Trips_2017_Q2.csv",
        DATA_RAW / "Divvy_Trips_2017_Q3.csv",
        DATA_RAW / "Divvy_Trips_2017_Q4.csv",
    ],
    stations_path=DATA_RAW / "Divvy_Stations_2017_Q3Q4 (2).csv",
    save_demand_to=DATA_PROCESSED / "divvy_hourly_demand_weather.csv",
)
demand.head()

Saved demand dataset: /Users/martynaharasym/Desktop/DSB/Personal_project/bike-sharing-rebalancing/data/processed/divvy_hourly_demand_weather.csv


,station_id,hour,departures,arrivals,net_demand,temp,tmin,tmax,rhum,prcp,snwd,wspd,pres
0,66,2017-01-01 00:00:00,0,0,0,-1.1,-6.2,4.9,65,0.0,0,8.7,1020.4
1,66,2017-01-01 01:00:00,0,0,0,-1.1,-6.2,4.9,65,0.0,0,8.7,1020.4
2,66,2017-01-01 02:00:00,0,0,0,-1.1,-6.2,4.9,65,0.0,0,8.7,1020.4
3,66,2017-01-01 03:00:00,0,0,0,-1.1,-6.2,4.9,65,0.0,0,8.7,1020.4
4,66,2017-01-01 04:00:00,0,0,0,-1.1,-6.2,4.9,65,0.0,0,8.7,1020.4


In [8]:
demand_model = add_time_and_lag_features(
    demand,
    lags=(1, 24, 168),
    drop_na_lags=True
)

demand_model.head()

,station_id,hour,departures,arrivals,net_demand,temp,tmin,tmax,rhum,prcp,...,dow_sin,dow_cos,month_sin,month_cos,dep_lag_1,arr_lag_1,dep_lag_24,arr_lag_24,dep_lag_168,arr_lag_168
168,2,2017-01-08 00:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,1.0,0.0,0.0,0.0,0.0
169,2,2017-01-08 01:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
170,2,2017-01-08 02:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
171,2,2017-01-08 03:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0
172,2,2017-01-08 04:00:00,0,0,0,-11.5,-15.8,-6.8,50,0.0,...,-0.781831,0.62349,0.5,0.866025,0.0,0.0,0.0,0.0,0.0,0.0


## Models

In [9]:
y_dep = demand_model["departures"]
y_arr = demand_model["arrivals"]

feature_cols = [
    "hour_sin","hour_cos",
    "dow_sin","dow_cos",
    "month_sin","month_cos",
    "is_weekend",
    "temp","tmin","tmax","rhum","prcp","snwd","wspd","pres",
    "dep_lag_1","dep_lag_24", 'dep_lag_168',
    "arr_lag_1","arr_lag_24", "arr_lag_168"
]

X = demand_model[feature_cols]

In [10]:
split_time = demand_model["hour"].quantile(0.8)

train = demand_model["hour"] <= split_time
test  = demand_model["hour"] > split_time

X_train = X[train]
X_test  = X[test]

y_dep_train = y_dep[train]
y_dep_test  = y_dep[test]

y_arr_train = y_arr[train]
y_arr_test  = y_arr[test]

In [11]:
SAMPLE_N = None 

dfm = demand_model.copy()

dfm = dfm.sort_values(["hour", "station_id"])

if SAMPLE_N is not None and SAMPLE_N < len(dfm):
    dfm = dfm.sample(SAMPLE_N, random_state=42).sort_values(["hour", "station_id"])

X = dfm[feature_cols]
y_dep = dfm["departures"]
y_arr = dfm["arrivals"]

split_time = dfm["hour"].quantile(0.8)

train_mask = dfm["hour"] <= split_time
test_mask  = dfm["hour"] > split_time

X_train, X_test = X[train_mask], X[test_mask]
y_dep_train, y_dep_test = y_dep[train_mask], y_dep[test_mask]
y_arr_train, y_arr_test = y_arr[train_mask], y_arr[test_mask]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


model_specs = {
    "HGBR": HistGradientBoostingRegressor(
        max_depth=8,
        learning_rate=0.08,
        max_iter=300,
        random_state=42
    ),

    "Ridge": Ridge(alpha=1.0, random_state=42),

   "Poisson": PoissonRegressor(alpha=1e-4, max_iter=300),

    "LightGBM": LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=200,
        max_depth=20,
        n_jobs=-1,
        random_state=42
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        n_jobs=-1,
        random_state=42
    ),
}

In [12]:
rows = []
results = []

import pandas as pd
from sklearn.base import clone

for name, model in model_specs.items():

    print(f"Training {name}...")

    # departures
    mdl_dep = clone(model)
    mdl_dep.fit(X_train, y_dep_train)
    pred_dep = mdl_dep.predict(X_test)

    # arrivals
    mdl_arr = clone(model)
    mdl_arr.fit(X_train, y_arr_train)
    pred_arr = mdl_arr.predict(X_test)

    pred_dep = np.clip(pred_dep, 0, None)
    pred_arr = np.clip(pred_arr, 0, None)

    pred_net = pred_arr - pred_dep
    true_net = y_arr_test - y_dep_test

    row = {
        "model": name,
        "dep_MAE": mean_absolute_error(y_dep_test, pred_dep),
        "dep_RMSE": rmse(y_dep_test, pred_dep),
        "arr_MAE": mean_absolute_error(y_arr_test, pred_arr),
        "arr_RMSE": rmse(y_arr_test, pred_arr),
        "net_MAE": mean_absolute_error(true_net, pred_net),
        "net_RMSE": rmse(true_net, pred_net),
    }

    results.append(row)

    # save partial results every iteration
    pd.DataFrame(results).to_csv("model_results_partial.csv", index=False)

    print(f"{name} finished.")

Training HGBR...
HGBR finished.
Training Ridge...
Ridge finished.
Training Poisson...


/Users/martynaharasym/Desktop/DSB/Personal_project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_glm/glm.py:290: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result(
/Users/martynaharasym/Desktop/DSB/Personal_project/.venv/lib/python3.13/site-packages/sklearn/linear_model/_glm/glm.py:290: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result(


Poisson finished.
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.079912 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1508
[LightGBM] [Info] Number of data points in the train set: 4021875, number of used features: 21
[LightGBM] [Info] Start training from score 0.847034
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061761 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1508
[LightGBM] [Info] Number of data points in the train set: 4021875, number of used features: 21
[LightGBM] [Info] Start training from score 0.846959
LightGBM finished.
Training XGBoost...
XGBoost finished.
Training ExtraTrees...
ExtraTrees finished.
Training RandomForest...
RandomForest finish

In [15]:
df_results = pd.DataFrame(results)
df_results

,model,dep_MAE,dep_RMSE,arr_MAE,arr_RMSE,net_MAE,net_RMSE
0,HGBR,0.362767,0.832453,0.367165,0.855796,0.415163,1.024828
1,Ridge,0.374209,0.945263,0.373062,0.969262,0.456264,1.099768
2,Poisson,0.590693,1.332268,0.588147,1.374714,0.435733,1.306669
3,LightGBM,0.366873,0.830641,0.366610,0.850196,0.407418,1.015516
4,XGBoost,0.364064,0.830204,0.367043,0.852977,0.404775,1.016965
5,ExtraTrees,0.373958,0.842541,0.375914,0.859161,0.409690,1.017997
6,RandomForest,0.372860,0.854403,0.374369,0.875373,0.414294,1.035944
